# Gate 3: удержание клиентов РКО

Работающий ноутбук: 5 сценариев, расчёт на сохранённых моделях и результатах. Учебная панель РКО и воздействия созданы генератором; UCI проверяется отдельно. Описания метода и ограничений — в `gate3-prototype.md`.

Запуск всех расчётов с нуля: `python prototype/run.py`. Для исполнения ноутбука установите `jupyter` в то же окружение. Сначала выполните ячейки по порядку.

In [1]:
from pathlib import Path
import sys, json, pickle
import pandas as pd
ROOT = Path.cwd() if (Path.cwd()/"core.py").exists() else Path.cwd()/"prototype"
sys.path.insert(0, str(ROOT))
from core import quote, recommend, select_campaign
with (ROOT/"models/bundle.pkl").open("rb") as f:
    models = pickle.load(f)
data = pd.read_csv(ROOT/"results/predictions.csv.gz")
metrics = json.loads((ROOT/"results/risk_metrics.json").read_text())
print("Клиентов в отложенном периоде:", len(data))
print("Точность в топ-10%:", round(metrics["any"]["ml"]["precision_at_10"], 4))

Клиентов в отложенном периоде: 7309
Точность в топ-10%: 0.6525


## 1. Скор → ожидаемый эффект → действие
Две задачи оттока и эффект трёх действий вычисляются сохранёнными моделями.

In [2]:
client = data[data.offer.gt(0)].sort_values("risk_any", ascending=False).iloc[:1].copy()
risk = models["risk"].predict(client)
uplift = models["uplift"].predict(client)
print(client.client_id.iloc[0])
print("Риски:", {k: round(float(v[0]), 4) for k, v in risk.items()})
print(recommend(client, uplift).to_string(index=False))

SYN-045180
Риски: {'close': 1.0, 'quiet': 1.0, 'any': 1.0}
 client_id  current_month  after_month  before_year  after_year  saving_year  budget_cost  allowed  offer  predicted_uplift  predicted_revenue
SYN-045180           4339         4169        52068       47058         5010         2970     True      1          0.650665       39039.916949


## 2. Оффер с ростом полной платы
Снижение базового тарифа может увеличить комиссии за платежи. Пакет может стать дорогим после промо.

In [3]:
for payments in [15, 50, 180]:
    for offer in [1, 2, 3]:
        result = quote("advanced", payments, offer)
        print("Платежи:", payments, "Оффер:", offer, "Экономия/год:", result["saving_year"], "Допустим:", result["allowed"])

Платежи: 15 Оффер: 1 Экономия/год: -7170 Допустим: False
Платежи: 15 Оффер: 2 Экономия/год: 13860 Допустим: True
Платежи: 15 Оффер: 3 Экономия/год: 4060 Допустим: True
Платежи: 50 Оффер: 1 Экономия/год: 5010 Допустим: True
Платежи: 50 Оффер: 2 Экономия/год: 5460 Допустим: True
Платежи: 50 Оффер: 3 Экономия/год: 4060 Допустим: True
Платежи: 180 Оффер: 1 Экономия/год: 5010 Допустим: True
Платежи: 180 Оффер: 2 Экономия/год: -25740 Допустим: False
Платежи: 180 Оффер: 3 Экономия/год: 4060 Допустим: True


## 3. Бюджет → список → случайный контроль
Модельные прогнозы в списке не являются измеренным эффектом кампании.

In [4]:
columns = ["client_id", "offer", "budget_cost", "saving_year", "predicted_uplift", "predicted_revenue"]
for budget in [300_000, 3_000_000]:
    chosen = select_campaign(data[columns], budget)
    treated = chosen[chosen.group.eq("treatment")]
    print("Бюджет", budget, "Выбрано", len(chosen), "Контроль", int(chosen.group.eq("control").sum()), "Расход", int(treated.budget_cost.sum()))
print(chosen.head(8).to_string(index=False))

Бюджет 300000 Выбрано 92 Контроль 10 Расход 262982
Бюджет 3000000 Выбрано 916 Контроль 92 Расход 2705824
 client_id  offer  budget_cost  saving_year  predicted_uplift  predicted_revenue     group
SYN-047260      1         2970         5010          0.786517       47190.990639 treatment
SYN-047801      1         2970         5010          0.740357       44421.417850 treatment
SYN-042030      1         2970         5010          0.730198       43811.908895 treatment
SYN-048452      1         2970         5010          0.729709       43782.523070   control
SYN-045250      1         2970         5010          0.721594       43295.616423 treatment
SYN-044756      1         2970         5010          0.715094       42905.663388 treatment
SYN-048245      1         2970         5010          0.688468       41308.108670 treatment
SYN-044585      1         2970         5010          0.674205       40452.274015 treatment


## 4. Исключение до модели и остановка при дрейфе
При запрете модели не вызываются.

In [5]:
from core import eligibility
raw = pd.read_csv(ROOT/"data/snapshots.csv.gz")
excluded = raw[raw.compliance].iloc[:1]
print("Клиент с ограничением допущен:", bool(eligibility(excluded).iloc[0]))
try:
    select_campaign(data[columns], drift_psi=.3)
except ValueError as e:
    print(str(e))
safety = json.loads((ROOT/"results/safety.json").read_text())
for case in safety:
    print(case["case"], case["title"], "Пройден" if case["passed"] else "Не пройден")

Клиент с ограничением допущен: False
Campaign blocked: distribution shift PSI > 0.2
1 Фильтр 115-ФЗ до модели Пройден
2 Сезонное падение не повышает скор Не пройден
3 Срез и зрелость таргета Пройден
4 Разрыв полноты по отраслям и регионам ≤5 п.п. Не пройден
5 Оффер не увеличивает текущую плату Пройден
6 Детерминированный расчёт экономии Пройден
7 Не более одного оффера за 30 дней Пройден
8 Переход в Точку исключён Пройден
9 Сдвиг распределения → сигнал и остановка Пройден


## 5. Ошибка и отдельные публичные наблюдения
Высокая вероятность допускает отсутствие события. Публичный набор относится к телекому и не проверяет офферы РКО.

In [6]:
error = json.loads((ROOT/"results/demo_error.json").read_text())
print("Ошибка:", error["client_id"], "риск", round(error["predicted_risk"], 4), "исход", error["observed_churn"])
pub = json.loads((ROOT/"results/public_metrics.json").read_text())
print("UCI: тест", pub["test"], "ML", pub["ml"]["precision_at_10"], "правила", pub["rules"]["precision_at_10"])
print(pd.read_csv(ROOT/"results/public_predictions.csv").sort_values("probability", ascending=False).head(5).to_string(index=False))

Ошибка: SYN-040090 риск 0.9259 исход 0
UCI: тест 795 ML 0.925 правила 0.625
 Call Failure  Complains  Subscription Length  Charge Amount  Seconds of Use  Frequency of use  Frequency of SMS  Distinct Called Numbers  Tariff Plan  Churn  probability  record
            3          1                    9              1            1487                19                 0                        7            1      1     0.999466    1049
            5          1                    9              1            1733                22                 7                        7            1      1     0.999026    2249
            0          1                    6              0            1068                 7                 0                        1            1      1     0.998779    1699
            6          1                   11              2            1602                24                 0                        6            1      1     0.998308    2849
           18          1     

## Воспроизведение
`python prototype/run.py` пересоздаёт панель, обучает модели и обновляет метрики. `python -m unittest discover -s prototype/tests -v` проверяет происхождение признаков, границы времени, бюджет, контроль, допуск и цену. `python prototype/app.py` открывает ту же модель через локальную демо на http://127.0.0.1:8765. Численные результаты при повторном обучении фиксированы seed; время исполнения меняется.